# GSE221156 (human pancreatic islets): load the libraries, extract the files, match the metadata

Step 1 of the analysis. It does four things: loads the packages, unpacks the archive, downloads the GEO sample records, and **matches every library to its donor, group (ND, PD, T2D) and clinical information**.
It stops with a library table you can reuse. Nothing is filtered or normalised here.

Key facts: 54 libraries (42 single-donor, 12 pools of two donors). Titles like `Islet70/71_MS19034` are **pools**: the cells of two donors in one library, so those libraries cannot be given one group.

## 1. Setup

**Step:** Load the packages and set the paths and options. Change only this cell.

In [ ]:
library(Seurat)
library(Matrix)

tar_file <- "C:/Users/user/Downloads/GSE221156_RAW.tar"
raw_dir  <- "C:/Users/user/Downloads/GSE221156_RAW"
out_dir  <- "GSE221156_results"

groups_to_keep <- c("ND", "T2D")   # used later; add "PD" for the pre-diabetic donors
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

## 2. Find the files

**Step:** Unpack the archive if needed, then list the libraries. Each library has three files: `_matrix.mtx.gz`, `_features.tsv.gz` and `_barcodes.tsv.gz`.

In [ ]:
if (length(list.files(raw_dir, pattern = "_matrix\\.mtx\\.gz$")) == 0) {
  dir.create(raw_dir, showWarnings = FALSE, recursive = TRUE)
  untar(tar_file, exdir = raw_dir)
}

matrix_files <- list.files(raw_dir, pattern = "_matrix\\.mtx\\.gz$", full.names = TRUE, recursive = TRUE)
libraries <- data.frame(file_prefix = sub("_matrix\\.mtx\\.gz$", "", matrix_files))
libraries$sample_id  <- basename(libraries$file_prefix)
libraries$gsm        <- sub("_.*$", "", libraries$sample_id)           # GSM6846488
libraries$library_id <- sub("^GSM[0-9]+_", "", libraries$sample_id)    # MS17001
nrow(libraries)

*What you should see:* 54 libraries.

## 3. Sample information from GEO

**Step:** Download the GEO sample records once (title and a text field with sex, BMI, age, HbA1c, chemistry and so on) and save them, so later runs work offline.

In [ ]:
geo_file <- file.path(out_dir, "geo_sample_metadata.csv")

if (file.exists(geo_file)) {
  geo <- read.csv(geo_file, stringsAsFactors = FALSE)
} else {
  if (!requireNamespace("GEOquery", quietly = TRUE)) BiocManager::install("GEOquery")
  gse <- GEOquery::getGEO("GSE221156", GSEMatrix = FALSE)
  records <- GEOquery::GSMList(gse)
  geo <- data.frame(
    gsm             = names(records),
    title           = sapply(records, function(x) GEOquery::Meta(x)$title[1]),
    characteristics = sapply(records, function(x) paste(GEOquery::Meta(x)$characteristics_ch1, collapse = " || "))
  )
  write.csv(geo, geo_file, row.names = FALSE)
}
head(geo[, c("gsm", "title")], 3)

**Step:** Split the text field into one column per item (`bmi`, `sex`, `chemistry`, ...).

In [ ]:
split_characteristics <- function(text) {
  items  <- strsplit(text, " \\|\\| ")[[1]]
  values <- trimws(sub("^[^:]*:", "", items))
  names(values) <- tolower(gsub("[^A-Za-z0-9]+", "_", sub(":.*$", "", items)))
  as.list(values)
}

parsed <- lapply(geo$characteristics, split_characteristics)
for (field in unique(unlist(lapply(parsed, names)))) {
  geo[[field]] <- sapply(parsed, function(x) if (is.null(x[[field]])) NA_character_ else x[[field]])
}
names(geo)[names(geo) == "library_id"] <- "geo_library_id"   # keep it apart from our own library_id

libraries <- merge(libraries, geo, by = "gsm", all.x = TRUE, sort = FALSE)
colnames(geo)

*What you should see:* Columns such as chemistry, disease_state, sex, bmi, race, hba1c, age and center.

## 4. Libraries and donors

**Step:** Which donor is in each library? A title like `Islet28` is one donor. A title like `Islet70/71` is a **pool** of two.

In [ ]:
donor_numbers <- regmatches(libraries$title, gregexpr("(?<=Islet)[0-9]+|(?<=/)[0-9]+", libraries$title, perl = TRUE))
libraries$donors <- sapply(donor_numbers, function(x) paste0("Islet", x, collapse = "+"))
libraries$pooled <- sapply(donor_numbers, length) > 1
libraries$donor  <- ifelse(libraries$pooled, NA, libraries$donors)

table(libraries$pooled)

*What you should see:* 42 single-donor and 12 pooled libraries.

**Step:** The donor groups, copied from the GEO series page (17 ND, 14 PD, 17 T2D).

In [ ]:
nd_ids  <- c(29, 34, 37, 38, 42, 45, 47, 54, 57, 59, 61, 68, 73, 80, 104, 107, 123)
pd_ids  <- c(40, 50, 53, 55, 56, 58, 62, 63, 70, 85, 89, 101, 118, 127)
t2d_ids <- c(28, 30, 31, 32, 33, 39, 41, 44, 48, 52, 60, 71, 84, 91, 126, 119, 128)

donor_groups <- data.frame(
  donor = paste0("Islet", c(nd_ids, pd_ids, t2d_ids)),
  group = rep(c("ND", "PD", "T2D"), c(length(nd_ids), length(pd_ids), length(t2d_ids)))
)
donor_groups$group_label <- c(ND = "No diabetes", PD = "Pre-diabetes", T2D = "Type 2 diabetes")[donor_groups$group]
libraries$group <- donor_groups$group[match(libraries$donor, donor_groups$donor)]   # NA for pools

**Step:** Which donors of the groups you keep have **no library of their own**? They sit only inside pools.

In [ ]:
donor_groups$own_library <- donor_groups$donor %in% libraries$donor
table(donor_groups$group, donor_groups$own_library)

only_in_pools <- donor_groups[!donor_groups$own_library & donor_groups$group %in% groups_to_keep, ]
only_in_pools

*What you should see:* All 17 ND donors have a library. For T2D, 14 do and Islet71, Islet84 and Islet119 do not.

**Step:** Save the library table (one row per library: accession, library id, donor or pool, group, chemistry and the clinical fields from GEO). The later steps read it back.

In [ ]:
library_table <- libraries[, c("gsm", "library_id", "title", "pooled", "donors", "donor", "group",
                               "chemistry", "disease_state", "sex", "age", "bmi", "hba1c", "race", "center")]
write.csv(library_table, file.path(out_dir, "library_table.csv"), row.names = FALSE)
head(library_table, 5)

*What you should see:* a table with 54 rows. Pooled libraries show two values joined by `/` (for example `bmi 32.9/31.7`) and an empty `donor` and `group`.

**Next:** build the donor table (BMI, lean or not lean), read the libraries into a Seurat object, then quality control. Those steps are sections 6 to 8 of `GSE221156_islet_ND_T2D_seurat.ipynb`.